# ELIQSIR — Step 1: Data Extraction

This notebook runs each extractor **independently** and saves the raw output to `data/raw/` as Parquet files that feed `02_transformation_and_loading.ipynb`.

| Step | Source | What is fetched |
|------|--------|-----------------|
| 0 | — | Pre-flight checks (env vars, MySQL connectivity, dump file) |
| 1 | — | Setup: paths & logging |
| 2 | **UniProt** | All reviewed human proteins (Swiss-Prot) |
| 3 | **ChEMBL** | `setup_database()` → bioactivity records for those proteins |
| 4 | **PDBe** | Best 3-D structures for proteins with ChEMBL activity |
| 5 | **PubMed** | Article abstracts for publications cited in ChEMBL |
| 6 | — | Extraction summary |

> **Run cells top-to-bottom.** Each section is independent enough to be re-run alone after a failure — Parquet files from previous sections are loaded from disk if they exist.


## 0 · Pre-flight Checks

Quickly verify that all required settings are present and that both MySQL databases are reachable **before** starting any long-running extraction.


In [5]:
import sys
from pathlib import Path

# ── make src/ importable without pip install ─────────────────────────────────
REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import mysql.connector

from src.config import settings
from src.extraction.chembl_extractor import discover_chembl_database
from src.utils.logging_config import get_logger

logger = get_logger("notebook.extraction")

# Shorthand for safe display of paths (project-relative, never absolute)
_dp = settings.display_path

RAW_DIR = REPO_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# ── 1. Required env vars ──────────────────────────────────────────────────────
checks = {
    "NCBI_EMAIL set":       bool(settings.ncbi_email),
    "MYSQL_USER set":       bool(settings.mysql_user),
    "MYSQL_PASSWORD set":   bool(settings.mysql_password),
}

# ── 2. ChEMBL SQLite database discovery ───────────────────────────────────────
chembl_version = None
chembl_db_path = None
try:
    chembl_version, chembl_db_path = discover_chembl_database(settings.chembl_dir)
    checks[f"ChEMBL SQLite v{chembl_version} found"] = True
except FileNotFoundError as exc:
    checks["ChEMBL SQLite database found"] = False
    chembl_msg = str(exc)

# ── 3. MySQL connectivity (warehouse only) ────────────────────────────────────
def _check_mysql(database=None) -> tuple[bool, str]:
    cfg = dict(
        host=settings.mysql_host,
        port=settings.mysql_port,
        user=settings.mysql_user,
        password=settings.mysql_password,
    )
    if database:
        cfg["database"] = database
    try:
        conn = mysql.connector.connect(**cfg)
        conn.close()
        return True, "OK"
    except Exception as exc:
        return False, str(exc)

mysql_ok, mysql_msg         = _check_mysql()
warehouse_ok, warehouse_msg = _check_mysql(settings.mysql_db)

checks["MySQL server reachable"]                     = mysql_ok
checks[f"Warehouse DB '{settings.mysql_db}' exists"] = warehouse_ok

# ── 4. Report ──────────────────────────────────────────────────────────────────
all_ok = True
for label, passed in checks.items():
    icon = "✅" if passed else "❌"
    print(f"  {icon}  {label}")
    if not passed:
        all_ok = False

if not all_ok:
    print("\n⚠  Fix the items above before running the extractors.")
    print(f"   MySQL server msg  : {mysql_msg}")
    print(f"   Warehouse DB msg  : {warehouse_msg}")
    if chembl_db_path is None:
        print(f"   ChEMBL dir        : {_dp(settings.chembl_dir)}")
        print("   → Download SQLite version from:")
        print("     https://ftp.ebi.ac.uk/pub/databases/chembl/ChEMBLdb/latest/")
    print(f"   NCBI email        : {'Present' if settings.ncbi_email else 'NOT SET'}")
else:
    print("\n✅  All checks passed — ready to extract.")
    print(f"\n   ChEMBL version    : {chembl_version}")
    print(f"   ChEMBL database   : {_dp(chembl_db_path)}")
    print(f"   MySQL host        : {settings.mysql_host}:{settings.mysql_port}")
    print(f"   Warehouse database: {settings.mysql_db}")
    print(f"   Raw output dir    : {_dp(RAW_DIR)}")
    print(f"   NCBI email        : {'Present' if settings.ncbi_email else 'NOT SET'}")

  ✅  NCBI_EMAIL set
  ✅  MYSQL_USER set
  ✅  MYSQL_PASSWORD set
  ✅  ChEMBL SQLite v36 found
  ✅  MySQL server reachable
  ❌  Warehouse DB 'eliqsir_dw' exists

⚠  Fix the items above before running the extractors.
   MySQL server msg  : OK
   Warehouse DB msg  : 1049 (42000): Unknown database 'eliqsir_dw'
   NCBI email        : Present


## 1 · Setup

Imports, paths, and logging — already done in the pre-flight cell above.  
This section is kept for reference; nothing extra to run.

---

## 2 · UniProt Extraction

`UniProtExtractor` queries the UniProt `/stream` endpoint and returns all **reviewed** (*Swiss-Prot*) human proteins in a single TSV request.

**Selection criteria:**
- Organism: *Homo sapiens* (taxonomy ID `9606`)
- Quality: reviewed (Swiss-Prot) only — manually curated, high confidence

**Columns returned:** `accession` · `gene_names` · `protein_name` · `organism_name` · `protein_sequence` · `protein_class` · `ec_number` · `catalyzed_reaction`


In [6]:
from src.extraction import UniProtExtractor

uniprot_ext = UniProtExtractor(
    organism_id=settings.uniprot_organism_id,   # 9606 = Homo sapiens
    reviewed=settings.uniprot_reviewed_only,    # Swiss-Prot only
)

df_raw_uniprot = uniprot_ext.extract()

print(f"Shape  : {df_raw_uniprot.shape}")
print(f"Columns: {list(df_raw_uniprot.columns)}")
display(df_raw_uniprot.head(5))


2026-03-26 04:09:30  INFO      src.extraction.uniprot_extractor  Fetching UniProt data – organism_id=9606, reviewed=True
2026-03-26 04:09:41  INFO      src.extraction.uniprot_extractor  UniProt extraction complete – 20431 proteins retrieved.
Shape  : (20431, 8)
Columns: ['accession', 'gene_names', 'protein_name', 'organism_name', 'protein_sequence', 'protein_class', 'ec_number', 'catalyzed_reaction']


,accession,gene_names,protein_name,organism_name,protein_sequence,protein_class,ec_number,catalyzed_reaction
0,A0A087X1C5,CYP2D7,Cytochrome P450 2D7 (EC 1.14.14.1),Homo sapiens (Human),MGLEALVPLAMIVAIFLLLVDLMHRHQRWAARYPPGPLPLPGLGNL...,Cytochrome P450 family,1.14.14.1,CATALYTIC ACTIVITY: Reaction=an organic molecu...
1,A0A096LP01,SMIM26 LINC00493,Small integral membrane protein 26,Homo sapiens (Human),MYRNEFTAWYRRMSVVYGIGTWSVLGSLLYYSRTMAKSSVDQKDGS...,SMIM26 family,NaN,NaN
2,A0A0B4J2F0,PIGBOS1,Protein PIGBOS1 (PIGB opposite strand protein 1),Homo sapiens (Human),MFRRLTFAQLLFATVLGIAGGVYIFQPVFEQYAKDQKELKEKMQLV...,NaN,NaN,NaN
3,A0A0C5B5G6,MT-RNR1,Mitochondrial-derived peptide MOTS-c (Mitochon...,Homo sapiens (Human),MRWQEMGYIFYPRKLR,NaN,NaN,NaN
4,A0A0K2S4Q6,CD300H,Protein CD300H (CD300 antigen-like family memb...,Homo sapiens (Human),MTQRAGAAMLPSALLLLCVPGCLTVSGPSTVMGAVGESLSVQCRYE...,CD300 family,NaN,NaN


In [7]:
out_path = RAW_DIR / "raw_uniprot.parquet"
df_raw_uniprot.to_parquet(out_path, index=False)
print(f"✓  Saved {len(df_raw_uniprot):,} rows → {_dp(out_path)}")


✓  Saved 20,431 rows → data/raw/raw_uniprot.parquet


## 3 · ChEMBL Bioactivity Data

Queries a **local SQLite** instance of [ChEMBL](https://www.ebi.ac.uk/chembl/) for bioactivity records
that link drugs to the UniProt proteins identified above.

The `ChemblExtractor` **auto-discovers** the ChEMBL version from the folder structure:
```
data/ChEMBL/
└── chembl_XX/                    # ← version detected from folder name
    └── chembl_XX_sqlite/
        └── chembl_XX.db          # ← SQLite database
```

**Download instructions:**
1. Go to https://ftp.ebi.ac.uk/pub/databases/chembl/ChEMBLdb/latest/
2. Download `chembl_XX_sqlite.tar.gz` (~1.5 GB)
3. Extract and move the `chembl_XX` folder into `data/ChEMBL/`

**Caching:** Results are cached as Parquet files in `data/ChEMBL/cache/` for fast subsequent loads.

Columns extracted (23 total):

| Column | Description |
|--------|-------------|
| `activity_id` | ChEMBL internal activity identifier |
| `drug_chembl_id` | ChEMBL compound ID (e.g. `CHEMBL25`) |
| `drug_name` | Preferred compound name |
| `molecule_type` | Small molecule / Antibody / etc. |
| `molecular_weight` | Molecular weight (g/mol) |
| `canonical_smiles` | Canonical SMILES string |
| `target_chembl_id` | ChEMBL target ID |
| `target_name` | Target preferred name |
| `organism` | Target organism |
| `standard_type` | Measurement type (IC50, Ki, …) |
| `standard_value` | Numeric measurement value |
| `standard_units` | Units (nM, µM, …) |
| `pchembl_value` | −log₁₀(activity) |
| `assay_type` | Assay type code |
| `assay_description` | Free-text assay description |
| `assay_organism` | Organism used in assay |
| `confidence_score` | Target assignment confidence (0–9) |
| `article_title` | Source article title |
| `journal` | Journal name |
| `year` | Publication year |
| `pubmed_id` | PubMed ID of source article |
| `doi` | DOI of source article |
| `uniprot_id` | UniProt accession (join key) |

In [8]:
# Reload modules to pick up code changes (dev convenience)
import importlib
import src.utils.logging_config
import src.extraction.chembl_extractor
import src.extraction

importlib.reload(src.utils.logging_config)
importlib.reload(src.extraction.chembl_extractor)
importlib.reload(src.extraction)

from src.extraction import ChemblExtractor

uniprot_accessions = df_raw_uniprot["accession"].dropna().unique().tolist()
print(f"Filtering ChEMBL for {len(uniprot_accessions):,} UniProt accessions …\n")

# Initialize extractor - auto-discovers ChEMBL version from folder structure
chembl_ext = ChemblExtractor(
    chembl_dir=settings.chembl_dir,
    use_cache=True,  # Cache results as Parquet for fast subsequent loads
)

# Optional: Print database summary
chembl_ext.print_database_info()

# Extract bioactivity data (with progress bar and caching)
df_raw_chembl = chembl_ext.extract(uniprot_ids=uniprot_accessions)

print(f"\nShape   : {df_raw_chembl.shape}")
print(f"Columns : {list(df_raw_chembl.columns)}")
display(df_raw_chembl.head(5))

# Save to raw output directory (separate from cache)
out_path = RAW_DIR / "raw_chembl.parquet"
df_raw_chembl.to_parquet(out_path, index=False)
print(f"\n✓  Saved {len(df_raw_chembl):,} rows → {_dp(out_path)}")

Filtering ChEMBL for 20,431 UniProt accessions …

2026-03-26 04:09:41  INFO      src.extraction.chembl_extractor  ============================================================
2026-03-26 04:09:41  INFO      src.extraction.chembl_extractor  ChEMBL SQLite Extractor initialized
2026-03-26 04:09:41  INFO      src.extraction.chembl_extractor    Version  : 36
2026-03-26 04:09:41  INFO      src.extraction.chembl_extractor    Database : data/ChEMBL/chembl_36/chembl_36_sqlite/chembl_36.db
2026-03-26 04:09:41  INFO      src.extraction.chembl_extractor    Cache dir: data/ChEMBL/cache
2026-03-26 04:09:41  INFO      src.extraction.chembl_extractor    Use cache: True
2026-03-26 04:09:41  INFO      src.extraction.chembl_extractor  ============================================================
✓ ChEMBL version 36 detected
  Database: data/ChEMBL/chembl_36/chembl_36_sqlite/chembl_36.db
  Size: 27.70 GB

ChEMBL v36 Database Summary
Database file: data/ChEMBL/chembl_36/chembl_36_sqlite/chembl_36.db
Size: 27

,activity_id,drug_chembl_id,drug_name,molecule_type,molecular_weight,canonical_smiles,target_chembl_id,target_name,organism,standard_type,...,assay_type,assay_description,assay_organism,confidence_score,article_title,journal,year,pubmed_id,doi,uniprot_id
0,12645440,CHEMBL2322194,None,Small molecule,445.55,NS(=O)(=O)OC[C@@H]1C[C@@H](N2CCc3c(N[C@H]4CCc5...,CHEMBL2321622,Ubiquitin-like modifier-activating enzyme 6,Homo sapiens,IC50,...,B,Inhibition of UBA6 (unknown origin),Homo sapiens,9,Exploring a new frontier in cancer treatment: ...,J Med Chem,2013.0,23360215.0,10.1021/jm301420b,A0AVT1
1,12645445,CHEMBL2017005,None,Small molecule,462.49,NS(=O)(=O)OC[C@H]1O[C@@H](n2cnc3c(N[C@H]4CCc5c...,CHEMBL2321622,Ubiquitin-like modifier-activating enzyme 6,Homo sapiens,IC50,...,B,Inhibition of UBA6 (unknown origin) in presenc...,Homo sapiens,9,Exploring a new frontier in cancer treatment: ...,J Med Chem,2013.0,23360215.0,10.1021/jm301420b,A0AVT1
2,18483955,CHEMBL1231160,PEVONEDISTAT,Small molecule,443.53,NS(=O)(=O)OC[C@@H]1C[C@@H](n2ccc3c(N[C@H]4CCc5...,CHEMBL2321622,Ubiquitin-like modifier-activating enzyme 6,Homo sapiens,IC50,...,B,Inhibition of UBA6 (unknown origin) assessed a...,Homo sapiens,9,Interrogating the Roles of Post-Translational ...,J Med Chem,2018.0,28505447.0,10.1021/acs.jmedchem.6b01817,A0AVT1
3,18574594,CHEMBL4226903,None,Small molecule,559.32,Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)(O)OP(=O)(O...,CHEMBL4295630,ADP-ribose glycohydrolase MACROD2,Homo sapiens,Kd,...,B,Binding affinity to human MDO2 by ITC,Homo sapiens,9,Adenosine analogs bearing phosphate isosteres ...,Bioorg Med Chem,2018.0,29501416.0,10.1016/j.bmc.2018.02.006,A1Z1Q3
4,24775879,CHEMBL1231160,PEVONEDISTAT,Small molecule,443.53,NS(=O)(=O)OC[C@@H]1C[C@@H](n2ccc3c(N[C@H]4CCc5...,CHEMBL2321622,Ubiquitin-like modifier-activating enzyme 6,Homo sapiens,IC50,...,B,Inhibition of His-tagged UBA6 (unknown origin)...,Homo sapiens,9,NAE modulators: A potential therapy for gastri...,Eur J Med Chem,2022.0,35131538.0,10.1016/j.ejmech.2022.114156,A0AVT1



✓  Saved 6,441,095 rows → data/raw/raw_chembl.parquet


In [9]:
# Summary statistics (already printed by extractor, but more detail here)
print("Unique drugs      :", df_raw_chembl["drug_chembl_id"].nunique())
print("Unique proteins   :", df_raw_chembl["uniprot_id"].nunique())
print("Unique PubMed IDs :", df_raw_chembl["pubmed_id"].nunique())
print("Articles with DOI :", df_raw_chembl["doi"].notna().sum())

print("\nMolecule types:")
display(df_raw_chembl["molecule_type"].value_counts().to_frame())

print("\nMeasurement types (top 10):")
display(df_raw_chembl["standard_type"].value_counts().head(10).to_frame())

# Save to raw output directory (separate from cache)
out_path = RAW_DIR / "raw_chembl.parquet"
df_raw_chembl.to_parquet(out_path, index=False)
print(f"\n✓  Saved {len(df_raw_chembl):,} rows → {_dp(out_path)}")

Unique drugs      : 1509932
Unique proteins   : 5327
Unique PubMed IDs : 36351
Articles with DOI : 2491984

Molecule types:


,count
molecule_type,
Small molecule,5249711
Unknown,361484
Protein,35203
Oligosaccharide,302
Oligonucleotide,69



Measurement types (top 10):


,count
standard_type,
Potency,2691362
IC50,1601949
Ki,544167
Inhibition,516884
EC50,200984
Kd,176101
AC50,165812
Activity,123828
Residual Activity,70152



✓  Saved 6,441,095 rows → data/raw/raw_chembl.parquet


## 4 · PDBe Extraction

`PdbeExtractor` queries the **PDBe Graph API** `/best_structures/{uniprot_id}` endpoint for each protein that appeared in the ChEMBL results.

The `/best_structures/` endpoint returns pre-ranked, high-quality structures — no need to filter by resolution manually.

**Key columns returned:** `uniprot_id` · `pdb_id` · `chain_id` · `resolution` · `coverage` · `method` · `unp_start` · `unp_end`

**Rate limiting:** `PDBE_REQUEST_DELAY_S` (default `0.1` s between requests) is applied to comply with EBI server guidelines.


In [10]:
from src.extraction import PdbeExtractor

# Only request structures for proteins that actually have ChEMBL activities
active_accessions = df_raw_chembl["uniprot_id"].dropna().unique().tolist()
print(f"Fetching PDB structures for {len(active_accessions):,} proteins …")
print("(This may take several minutes — 0.1 s rate-limit per request)")

pdbe_ext = PdbeExtractor(
    request_delay_s=settings.pdbe_request_delay_s,
    timeout_s=settings.pdbe_timeout_s,
)
df_raw_pdbe = pdbe_ext.extract(uniprot_ids=active_accessions)

print(f"\nShape   : {df_raw_pdbe.shape}")
print(f"Columns : {list(df_raw_pdbe.columns)}")
display(df_raw_pdbe.head(10))


Fetching PDB structures for 5,327 proteins …
(This may take several minutes — 0.1 s rate-limit per request)
2026-03-26 04:30:33  INFO      src.extraction.pdbe_extractor  Starting PDB structure fetch for 5327 unique proteins.
2026-03-26 04:31:09  INFO      src.extraction.pdbe_extractor  Progress: 100 / 5327 proteins processed – 1620 structures collected.
2026-03-26 04:31:47  INFO      src.extraction.pdbe_extractor  Progress: 200 / 5327 proteins processed – 4059 structures collected.
2026-03-26 04:32:24  INFO      src.extraction.pdbe_extractor  Progress: 300 / 5327 proteins processed – 5559 structures collected.
2026-03-26 04:32:59  INFO      src.extraction.pdbe_extractor  Progress: 400 / 5327 proteins processed – 8380 structures collected.
2026-03-26 04:33:34  INFO      src.extraction.pdbe_extractor  Progress: 500 / 5327 proteins processed – 9947 structures collected.
2026-03-26 04:34:10  INFO      src.extraction.pdbe_extractor  Progress: 600 / 5327 proteins processed – 11912 structures

,uniprot_id,pdb_id,chain_id,resolution,coverage,unp_start,unp_end,method
0,A0AVT1,7pvn,A,2.71,1.0,1,1052,X-ray diffraction
1,A0AVT1,7pvn,B,2.71,1.0,1,1052,X-ray diffraction
2,A0AVT1,9qh5,B,3.09,1.0,1,1052,Electron Microscopy
3,A0AVT1,9qic,B,3.29,1.0,1,1052,Electron Microscopy
4,A0AVT1,9qiv,B,3.44,1.0,1,1052,Electron Microscopy
5,A0AVT1,9qim,B,3.57,1.0,1,1052,Electron Microscopy
6,A0AVT1,9qgw,B,3.62,1.0,1,1052,Electron Microscopy
7,A0AVT1,9qig,B,3.94,1.0,1,1052,Electron Microscopy
8,A0AVT1,9qii,B,3.99,1.0,1,1052,Electron Microscopy
9,A0AVT1,9qip,B,4.15,1.0,1,1052,Electron Microscopy


In [11]:
# Coverage & resolution overview
print("Proteins with at least one structure :", df_raw_pdbe["uniprot_id"].nunique())
print("Unique PDB entries                   :", df_raw_pdbe["pdb_id"].nunique())
print("\nExperimental methods:")
display(df_raw_pdbe["method"].value_counts().to_frame())

print("\nResolution (Å) — lower is better:")
display(df_raw_pdbe["resolution"].describe().to_frame().T)

# Save
out_path = RAW_DIR / "raw_pdbe.parquet"
df_raw_pdbe.to_parquet(out_path, index=False)
print(f"\n✓  Saved {len(df_raw_pdbe):,} rows → {_dp(out_path)}")


Proteins with at least one structure : 4038
Unique PDB entries                   : 61839

Experimental methods:


,count
method,
X-ray diffraction,104446
Electron Microscopy,60163
Solution NMR,3211
Solid-state NMR,392
Electron crystallography,65
Neutron Diffraction,60
X-ray solution scattering,53
X-ray powder diffraction,15
EPR,1



Resolution (Å) — lower is better:


,count,mean,std,min,25%,50%,75%,max
resolution,164802.0,2.763824,1.688771,0.6,2.0,2.57,3.13,53.0



✓  Saved 168,406 rows → data/raw/raw_pdbe.parquet


## 5 · PubMed Extraction

`PubMedExtractor` uses **Biopython's `Bio.Entrez`** to batch-fetch article metadata from NCBI PubMed via the E-utils XML API.

**Batching strategy:** IDs are sent in groups of `NCBI_BATCH_SIZE` (default `200`) to stay within NCBI limits and keep XML responses manageable.

**Columns returned (7 total):**

| Column | Description |
|--------|-------------|
| `pubmed_id` | PubMed article identifier |
| `abstract` | Full article abstract text |
| `authors` | Semicolon-separated author list (`Last, First`) |
| `pub_date` | Publication date string |
| `year` | Publication year (integer) |
| `month` | Publication month (integer) |
| `doi` | Digital Object Identifier |


> `NCBI_EMAIL` must be set in `.env` — NCBI requires an email address to identify API callers.


In [12]:
from src.extraction import PubMedExtractor

# Use only the PubMed IDs that appeared in the ChEMBL results
pmids = (
    df_raw_chembl["pubmed_id"]
    .dropna()
    .unique()
    .tolist()
)
# Cast to str — Entrez expects string IDs
pmids = [str(int(p)) for p in pmids]
print(f"Fetching abstracts for {len(pmids):,} PubMed articles …")

pubmed_ext = PubMedExtractor(
    email=settings.ncbi_email,
    batch_size=settings.ncbi_batch_size,
    request_delay_s=settings.ncbi_request_delay_s,
)
df_raw_pubmed = pubmed_ext.extract(pubmed_ids=pmids)

print(f"\nShape   : {df_raw_pubmed.shape}")
print(f"Columns : {list(df_raw_pubmed.columns)}")
display(df_raw_pubmed.head(10))


Fetching abstracts for 36,351 PubMed articles …
2026-03-26 05:12:00  INFO      src.extraction.pubmed_extractor  Fetching abstracts for 36351 PubMed articles in batches of 200.
2026-03-26 05:12:06  INFO      src.extraction.pubmed_extractor  Progress: 200 / 36351 abstracts retrieved.
2026-03-26 05:12:08  INFO      src.extraction.pubmed_extractor  Progress: 400 / 36351 abstracts retrieved.
2026-03-26 05:12:10  INFO      src.extraction.pubmed_extractor  Progress: 600 / 36351 abstracts retrieved.
2026-03-26 05:12:13  INFO      src.extraction.pubmed_extractor  Progress: 800 / 36351 abstracts retrieved.
2026-03-26 05:12:15  INFO      src.extraction.pubmed_extractor  Progress: 1000 / 36351 abstracts retrieved.
2026-03-26 05:12:17  INFO      src.extraction.pubmed_extractor  Progress: 1200 / 36351 abstracts retrieved.
2026-03-26 05:12:19  INFO      src.extraction.pubmed_extractor  Progress: 1400 / 36351 abstracts retrieved.
2026-03-26 05:12:22  INFO      src.extraction.pubmed_extractor  Progress

,pubmed_id,abstract,authors,pub_date,year,month,doi
0,23360215,The labeling of proteins with small ubiquitin ...,"da Silva, Sara R; Paiva, Stacey-Lynn; Lukkaril...",2013-3-28,2013,3.0,10.1021/jm301420b
1,28505447,Post-translational modifications (PTMs) allot ...,"Buuh, Zakey Yusuf; Lyu, Zhigang; Wang, Rongshe...",2018-4-26,2018,4.0,10.1021/acs.jmedchem.6b01817
2,29501416,The human O-acetyl-ADP-ribose deacetylase MDO1...,"Zhang, Yuezhou; Jumppanen, Mikael; Maksimainen...",2018-5-01,2018,5.0,10.1016/j.bmc.2018.02.006
3,35131538,Neural precursor cell expressed developmentall...,"Liang, Qi; Liu, Maoyu; Li, Jian; Tong, Rongshe...",2022-3-05,2022,3.0,10.1016/j.ejmech.2022.114156
4,35597097,"A series of amino acid based 7H-pyrrolo[2,3-d]...","Sherrill, Lavinia M; Joya, Elva E; Walker, Ann...",2022-8-01,2022,8.0,10.1016/j.bmc.2022.116788
5,34748351,Accumulation of very long chain fatty acids (V...,"Come, Jon H; Senter, Timothy J; Clark, Michael...",2021-12-23,2021,12.0,10.1021/acs.jmedchem.1c00944
6,29928781,Erythropoietin-producing hepatocellular (EPH) ...,"Tröster, Alix; Heinzlmeir, Stephanie; Berger, ...",2018-8-20,2018,8.0,10.1002/cmdc.201800398
7,28408219,A structure-activity relationship has been dev...,"Shaw, Simon J; Goff, Dane A; Lin, Nan; Singh, ...",2017-6-01,2017,6.0,10.1016/j.bmcl.2017.03.037
8,28256837,The plant Gymnema sylvestre has been used wide...,"Capolupo, Angela; Esposito, Roberta; Zampella,...",2017-4-28,2017,4.0,10.1021/acs.jnatprod.6b00793
9,30929949,To identify new potential therapeutic targets ...,"Crane, Erika A; Heydenreuter, Wolfgang; Beck, ...",2019-6-15,2019,6.0,10.1016/j.bmc.2019.03.022


In [13]:
abstracts_found = df_raw_pubmed["abstract"].notna() & (df_raw_pubmed["abstract"].str.strip() != "")
print(f"Articles with non-empty abstract : {abstracts_found.sum():,} / {len(df_raw_pubmed):,}")
print(f"Average abstract length (chars)  : {df_raw_pubmed['abstract'].dropna().str.len().mean():.0f}")
print(f"Articles with authors            : {(df_raw_pubmed['authors'] != '').sum():,}")
print(f"Articles with DOI                : {(df_raw_pubmed['doi'] != '').sum():,}")
print(f"Articles with pub_date           : {(df_raw_pubmed['pub_date'] != '').sum():,}")

print("\nYear distribution (top 10):")
display(df_raw_pubmed["year"].value_counts().head(10).sort_index().to_frame())

# Save
out_path = RAW_DIR / "raw_pubmed.parquet"
df_raw_pubmed.to_parquet(out_path, index=False)
print(f"\n✓  Saved {len(df_raw_pubmed):,} rows → {_dp(out_path)}")


Articles with non-empty abstract : 34,775 / 35,551
Average abstract length (chars)  : 899
Articles with authors            : 35,551
Articles with DOI                : 35,493
Articles with pub_date           : 35,551

Year distribution (top 10):


,count
year,
2010,1659
2011,1591
2012,1775
2013,1677
2016,1601
2017,1768
2018,1672
2019,1658
2020,1657



✓  Saved 35,551 rows → data/raw/raw_pubmed.parquet


## 6 · Extraction Summary

A consolidated view of what was extracted from each source.


In [14]:
summary = pd.DataFrame([
    {"source": "UniProt",  "rows": len(df_raw_uniprot), "file": "raw/raw_uniprot.parquet"},
    {"source": "ChEMBL",   "rows": len(df_raw_chembl),  "file": "raw/raw_chembl.parquet"},
    {"source": "PDBe",     "rows": len(df_raw_pdbe),    "file": "raw/raw_pdbe.parquet"},
    {"source": "PubMed",   "rows": len(df_raw_pubmed),  "file": "raw/raw_pubmed.parquet"},
])
display(summary.style.set_caption("Extraction Summary").hide(axis="index"))

print("\n✓  All raw Parquet files written to data/raw/")
print("   → Continue in notebooks/02_transformation_and_loading.ipynb")


source,rows,file
UniProt,20431,raw/raw_uniprot.parquet
ChEMBL,6441095,raw/raw_chembl.parquet
PDBe,168406,raw/raw_pdbe.parquet
PubMed,35551,raw/raw_pubmed.parquet



✓  All raw Parquet files written to data/raw/
   → Continue in notebooks/02_transformation_and_loading.ipynb
